# POD for 0D Isothermal Chemical Reactor
### Importing libraries

In [1]:
import os
import re
import sys
import time
import pyDOE
import itertools
import numpy as np
import joblib as jl
import pandas as pd
from tqdm import tqdm
import tensorflow as tf
import matplotlib.pyplot as plt
plt.style.use('paper')

from PCAfold import PCA

# sys.path.append(os.path.normpath(os.environ['WORKSPACE_PATH'] + '/ML/PrODE/prode.22.01/'))
from physics.data_generator import DataGenerator

In [2]:
# Set up case directory
path = './RVE/'
if not os.path.exists(path):
    os.makedirs(path)

# Number of cores available
CPUs = jl.cpu_count()

### Generating Training and Test Data Set
Input parameters

In [3]:
print(f'Generate Data')

# Training/Testing data
n_cases = 2000

# Physical model
# =============================================================================
# Input space (limit values)
input_space = {
    'Ttr': [8.e3,   1.5e4],        # Tanslational temperature [K]
    'T0':  [1.e3,   8.e3],         # Initial temperature [K]
    'P0':  [1.e3,   1.e4],         # Initial pressure [Pa]
    'XO2': [0.05e0, 1.e0]          # Initial O2 mole fraction
}

# Time (limit values)
time_limit = [1.e-14, 1.e-1]

# Timescales thermochemistry --------------------------------------------------
groups = ['RVE1', 'RVE3', 'RVE9', 'RVE18']

groups_graph = {
    0: {
        0: {
            0: [0,1],
            1: [2,3],
            2: [4,5],
        },
        1: {
            3: [6,7],
            4: [8,9],
            5: [10,11],
        },
        2: {
            6: [12,13],
            7: [14,15],
            8: [16,17],
        }
    }
}

# Thermochemistry -------------------------------------------------------------
species = {
    'O2': {
        'rovib': {
            'system': 'O3',
            'PES': 'UMN',
            'grouping': 'RVE18'
        }
    },
    'O': None
}

process_flags = {      # Reactive processes
    'excit': 1,
    'diss':  1
}

Solve ODEs

In [4]:
dataGen = DataGenerator(
    path,
    n_cases,
    time_limit,
    input_space,
    species,
    process_flags,
    groups_graph
)

ic = dataGen.initial_cond()
cases = dataGen.generate(ic)

# Extract data
t = cases[0][0]
y_cases = [case[1] for case in cases]

  1%|          | 19/2000 [00:00<00:10, 189.66it/s]

Generating initial conditions ...


  0%|          | 0/2000 [00:00<?, ?it/s]

Generating data ...


100%|██████████| 2000/2000 [06:06<00:00,  5.45it/s]


### Eigenmodes Analysis
Input parameters

In [18]:
split = True      # Split variables
scale = 'log'     # Transform data
nPC_max = 64      # Max number of preserved principal components

scalings = ['none', 'auto', 'std', 'pareto', 'vast', 'range', '0to1']

Set up PCA case directory

In [19]:
pca_dir = path + 'pca_' + scale
if split:
    pca_dir += '_split'
if not os.path.exists(pca_dir):
    os.makedirs(pca_dir)

Get split feature indeces for separate PCA 

In [20]:
def split_indeces(ls, indeces=[]):
        try:
            sub_ls = []
            start = 0
            for elem in ls:
                sub_ls.append(list(range(start,start+len(elem))))
                start = sub_ls[-1][-1] + 1
            indeces.append(sub_ls)
            ls = list(itertools.chain(*ls))
            return split_indeces(ls, indeces=indeces)
        except:
            return indeces

if split:
    groups_nested_list = dataGen.graph_to_nested_list(groups_graph)
    indeces = [[[0,1]]] + split_indeces(groups_nested_list)

Define error functions

In [21]:
reductions = ['max', 'mean']

def mape(y_true, y_pred, reduction='mean', axis=None):
    err = 100*np.abs((y_true - y_pred) / y_true)
    if reduction =='max':
        red_fn = np.amax
    else:
        red_fn = np.mean
    return red_fn(err, axis=axis)

def mse(y_true, y_pred, reduction='mean', axis=None):
    err = (y_true - y_pred)**2
    if reduction =='max':
        red_fn = np.amax
    else:
        red_fn = np.mean
    return red_fn(err, axis=axis)

def get_err(name):
    if name == 'mape':
        return mape
    elif name == 'mse':
        return mse

Concatenate the data matrix

In [22]:
print(f'\nConcatenate Data')

# Define the splitting function
split_fn = lambda y_i, i: [np.take(y_i, j, axis=1) for j in i]

# Define the concatenation functions
def concatenate(data, first):
    Y_ = None
    for data_i in data:
        # Extract data
        if first:
            if split:
                y = list(map(split_fn, data_i, indeces))
            else:
                y = [[y_i] for y_i in data_i]
        else:
            y = data_i
        # Concatenate data
        if Y_ is None:
            Y_ = y
        else:
            Y_ = tf.nest.map_structure(lambda Y_i, y_i: np.hstack((Y_i, y_i)), Y_, y)
    return Y_

def parallel_concatenate(data, first=True):
    # Prepocess indeces of the data list
    n = len(data)
    ratio = max(20,int(np.ceil(n/CPUs)))
    start_ends = []
    for c in range(CPUs):
        start_ends.append([ratio*c,ratio*(c+1)])
        if start_ends[-1][-1] >= n:
            start_ends[-1][-1] = n
            break
    data = [data[s:e] for (s, e) in start_ends]
    # Concatenate arrays
    print(f'>> Concatenate data list: n = {n}')
    data_concat = jl.Parallel(n_jobs=-1)(
        jl.delayed(concatenate)(data_i, first) for data_i in tqdm(data)
    )
    if len(data_concat) == 1:
        return data_concat[-1]
    else:
        return parallel_concatenate(data_concat, first=False)

# Get concatenated data
Y = parallel_concatenate(y_cases)

  0%|          | 0/96 [00:00<?, ?it/s]

>> Concatenate data list: n = 2000


100%|██████████| 5/5 [00:00<00:00, 1461.94it/s]

>> Concatenate data list: n = 96



100%|██████████| 1/1 [00:00<00:00, 348.51it/s]

>> Concatenate data list: n = 5


#### Eigenvalue decomposition

In [ ]:
print(f'\nEigenvalue decomposition')

eig_min = 1.e-14   # Minimum eigenvalue to be plotted

# Define directory
eig_dir = pca_dir + '/eigen'
if not os.path.exists(eig_dir):
    os.makedirs(eig_dir)

nPC = {}
for i, y_i in enumerate(Y):
    print(f'>> POD for {groups[i]} coarse-grained model dynamics')
    print(f'>>    Shapes:', tf.nest.map_structure(lambda x: x.shape, y_i))
    nPC[groups[i]] = []
    for j, y_ij in enumerate(y_i):
        for s, sc in enumerate(scalings):
            pca = PCA(
                np.log10(y_ij) if scale == 'log' else y_ij,
                scaling=sc,
                n_components=nPC_max,
                use_eigendec=True,
                nocenter=False
            )
            if s == 0:
                L = pca.L.reshape(-1,1)
            else:
                L = np.hstack((L, pca.L.reshape(-1,1)))
        nPC_cg = max(np.argmin(np.abs(L - eig_min), axis=0)) + 1
        nPC[groups[i]].append(nPC_cg)

        fig = plt.figure()
        # Y axis
        plt.ylim([eig_min, None])
        plt.ylabel('Eigenvalue')
        plt.yscale('log')
        # X axis
        plt.xlabel('Component Index')
        interval = int(np.ceil(nPC_cg/10))
        plt.xticks(np.arange(0,nPC_cg,interval), labels=np.arange(1,nPC_cg+1,interval))
        plt.xlim([0,nPC_cg-1])
        # Plot
        plt.plot(L, label=scalings)
        plt.legend()
        # Save figure
        fig_path = eig_dir+'/'+groups[i]
        if not os.path.exists(fig_path):
            os.makedirs(fig_path)
        plt.savefig(fig_path+'/split'+str(j+1)+'.pdf')
        plt.close()


>> POD for RVE1 coarse-grained model dynamics
>>    Shapes: [(10000, 4000)]


findfont: Font family ['serif'] not found. Falling back to DejaVu Sans.
findfont: Generic family 'serif' not found because none of the following families were found: Arial
findfont: Font family ['cursive'] not found. Falling back to DejaVu Sans.
findfont: Generic family 'cursive' not found because none of the following families were found: Arial
findfont: Font family ['Charter'] not found. Falling back to DejaVu Sans.
findfont: Font family ['monospace'] not found. Falling back to DejaVu Sans.
findfont: Generic family 'monospace' not found because none of the following families were found: Arial
findfont: Font family ['Charter'] not found. Falling back to DejaVu Sans.
findfont: Font family ['Charter'] not found. Falling back to DejaVu Sans.
findfont: Font family ['sans'] not found. Falling back to DejaVu Sans.
findfont: Generic family 'sans' not found because none of the following families were found: Arial


>> POD for RVE3 coarse-grained model dynamics
>>    Shapes: [(10000, 6000)]


#### Evaluate error and visualize the principal components

In [ ]:
print(f'\nEvaluate error and PC')

# Define the scaling method for the data matrix
scaling ='0to1'

# Define the error minimum value and the list of errors
err_min = 1.e-1
err_list = {'mape', 'mse'}

# Define directories
err_dir = pca_dir + '/err'
if not os.path.exists(err_dir):
    os.makedirs(err_dir)

pc_dir = pca_dir + '/pc'
if not os.path.exists(pc_dir):
    os.makedirs(pc_dir)

def pca_error(y, index, nPC):
    # Initilize error arrays
    err = {
        name: {
            r: np.zeros((1, len(scalings))) for r in reductions
        } for name in err_list
    }
    # Initialize the PCA object
    pca = PCA(
        np.log10(y) if scale == 'log' else y,
        scaling=scaling,
        n_components=nPC,
        use_eigendec=True,
        nocenter=False
    )
    # Calculate the principal components
    pc = pca.transform(np.log10(y) if scale == 'log' else y)
    # Calculate the reconstructed variables
    y_rec = pca.reconstruct(pc)
    y_rec = np.power(10,y_rec) if scale == 'log' else y_rec
    # Calculate errors
    for name in err_list:
        for r in reductions:
            err[name][r] = get_err(name)(y, y_rec, reduction=r)
    return err

def pca_modes(y, nPC, fig_path, nPC_plotted=16):
    # Initialize the PCA object
    pca = PCA(
        np.log10(y) if scale == 'log' else y,
        scaling=scaling,
        n_components=nPC,
        use_eigendec=True,
        nocenter=False
    )
    # Calculate the principal components
    pc = pca.transform(np.log10(y) if scale == 'log' else y)
    # Plot principal components
    for p, phi in enumerate(pc.T):
        if p == nPC_plotted:
            break
        fig = plt.figure()
        # Y axis
        plt.ylabel(r'$\psi_{{%s}}$' % str(p+1))
        # X axis
        plt.xlabel('t [s]')
        plt.xscale('log')
        plt.xlim(time_limit)
        # Plot
        plt.plot(t, phi)
        # Save figure
        plt.savefig(fig_path+'/phi'+str(p+1).zfill(3)+'.pdf')
        plt.close()

for i, y_i in enumerate(Y):
    print(f'>> Reconstructed PCA error for {groups[i]} coarse-grained model dynamics')
    for j, y_ij in enumerate(y_i):
        nPC_ij = int(min(nPC[groups[i]][j],nPC_max))
        # Errors --------------------------------------------------------------
        # Reconstruct variables for an increasing number of principal components
        err_ij = jl.Parallel(n_jobs=-1)(
            jl.delayed(pca_error)(y_ij, k, nPC_k) \
                for k, nPC_k in enumerate(tqdm(range(1, nPC_ij+1)))
        )
        err = err_ij[0]
        for e in err_ij:
            for name in err_list:
                for r in reductions:
                    err[name][r] = np.vstack((err[name][r],e[name][r]))
        # Define the max number of principal components to get small error
        PC_min = {
            name: {
                r: max(np.argmin(np.abs(err[name][r] - err_min), axis=0))+1 \
                    for r in reductions
            } for name in err_list
        }
        # Plot
        n, m = 2, 2
        fig, axs = plt.subplots(n, m, sharex=True, figsize=(12,10))
        for n_ in range(n):
            for m_ in range(m):
                if n_ == n-1:
                    err_, PC_ = err['mape'], PC_min['mape']
                    axs[n_,0].set_ylabel('MAPE')
                else:
                    err_, PC_ = err['mse'], PC_min['mse']
                    axs[n_,0].set_ylabel('MSE')
                # Text
                axs[n_,m_].text(
                    0.5, 0.05,
                    f'PC>{PC_[reductions[m_]]} for err<0.1',
                    horizontalalignment='center',
                    verticalalignment='center',
                    transform=axs[n_,m_].transAxes
                )
                # X axis
                axs[n_,m_].set_xlim([0,nPC_ij-1])
                interval = int(np.ceil(nPC_ij/10))
                axs[n_,m_].set_xticks(np.arange(0,nPC_ij,interval))
                axs[n_,m_].set_xticklabels(np.arange(1,nPC_ij+1,interval))
                # Y axis
                axs[n_,m_].set_yscale('log')
                if n_ == n-1:
                    axs[n_,m_].set_xlabel('# of Preserved Components')
                # Plot
                axs[n_,m_].plot(err_[reductions[m_]], label=scalings)
                axs[n_,m_].legend(fontsize=12)
        axs[0,0].set_title('Max')
        axs[0,1].set_title('Mean')
        plt.tight_layout()
        # Save figure
        fig_path = err_dir+'/'+groups[i]
        if not os.path.exists(fig_path):
            os.makedirs(fig_path)
        plt.savefig(fig_path+'/split'+str(j+1)+'.pdf')
        plt.close()
        
        # Principal components ------------------------------------------------
        fig_path = pc_dir+'/'+groups[i]+'/split'+str(j+1)
        if not os.path.exists(fig_path):
            os.makedirs(fig_path)
        pca_modes(y_ij, nPC_ij, fig_path)


#### Reconstruct variables

In [1]:
print(f'\nReconstruct variables')

# Max number of preserved principal components
nPC_list = [8, 16, 32]

# Define the scaling method for the data matrix
scaling ='0to1'

# Define directories
rec_dir = pca_dir + '/rec_var'
if not os.path.exists(rec_dir):
    os.makedirs(rec_dir)

def pca_reconstruct_var(y, nPC, fig_path, index):
    fig_path = fig_path+'/split'+str(index+1)
    if not os.path.exists(fig_path):
        os.makedirs(fig_path)
    # Initialize the PCA object
    pca = PCA(
        np.log10(y) if scale == 'log' else y,
        scaling=scaling,
        n_components=nPC,
        use_eigendec=True,
        nocenter=False
    )
    # Calculate the principal components
    pc = pca.transform(np.log10(y) if scale == 'log' else y)
    # Calculate the reconstructed variables
    y_rec = pca.reconstruct(pc)
    y_rec = np.power(10,y_rec) if scale == 'log' else y_rec
    # Plot comparison
    indices = np.random.choice(y.shape[1], size=20, replace=False)
    y_ = np.take(y, indices, axis=1)
    y_rec_ = np.take(y_rec, indices, axis=1)
    for i in range(y_.shape[1]):
        fig = plt.figure()
        # Y axis
        plt.ylabel(r'$Y$')
        plt.yscale('log')
        # X axis
        plt.xlabel(r'$t$ [s]')
        plt.xscale('log')
        plt.xlim(time_limit)
        # Plot
        plt.plot(t, y_.T[i], label='True')
        plt.plot(
            t, y_rec_.T[i],
            marker='^',
            fillstyle='none',
            markersize=5,
            markevery=y_rec_.shape[0]//20,
            ls='--',
            label='Reconstructed'
        )
        plt.legend()
        # Save figure
        plt.savefig(fig_path+'/case'+str(i+1).zfill(3)+'.pdf')
        plt.close()

for nPC_i in nPC_list:
    fig_path = rec_dir+'/pc_'+str(nPC_i).zfill(2)
    for i, y_i in enumerate(Y):
        print(f'>> Reconstruct variables for {groups[i]} coarse-grained model dynamics')
        fig_path_i = fig_path + '/' + groups[i]
        y_rec_ij = jl.Parallel(n_jobs=-1)(
            jl.delayed(pca_reconstruct_var)(y_ij, nPC_max, fig_path_i, j) \
                for j, y_ij in enumerate(tqdm(y_i))
        )


Reconstruct variables


NameError: name 'pca_dir' is not defined